### Upload or copy your five target catchment GIS layers: 

a. biophysical_table.csv

1. dawasamu_dem.tif
2. dawasama_catchment_boundary.shp
3. dawasamu_lulc.tif
4. dawasamu_erositivity_r.tif
5. dawasamu_erodibility_k.tif

Open your generated biophysical_table.csv and ensure the numbers in the lucode column match the exact integer pixel values used inside your land-use raster (dawasamu_lulc.tif).

Re-run your main script cell. It will process your spatial files and export your structural sediment erosion datasets into your outputs folder.

#### 1. Digital Elevation Model (DEM) - File Name in Script: dawasamu_dem.tif
Format: Single-band GeoTIFF (.tif)  - What it represents: The topography of the catchment.Critical Requirement: This raster must be hydrologically correct. Use GIS software or a python library to fill any artificial sinks or depressions before uploading, ensuring continuous downhill water routing.🌳 

In [1]:
pip install rasterio geopandas shapely

    kubernetes (>=9.0.0a1.0) ; extra == 'all_extras'
               ~~~~~~~~~~^
Note: you may need to restart the kernel to use updated packages.


In [16]:
import geopandas as gpd
import rasterio
from shapely.geometry import box

# 1. Open the raster and extract its bounding coordinates
raster_path = "1-dawasamu-dem.tif"

with rasterio.open(raster_path) as src:
    bounds = src.bounds  # BoundingBox(left, bottom, right, top)
    crs = src.crs  # Coordinate reference system of dem.tif

# 2. Create a Shapely polygon box representing the bounding box
bbox_polygon = box(bounds.left, bounds.bottom, bounds.right, bounds.top)

# 3. Create a GeoDataFrame with the original raster projection
gdf = gpd.GeoDataFrame({"geometry": [bbox_polygon]}, crs=crs)

# (Optional) Reproject to WGS84 (EPSG:4326) for standard lat/lon GeoJSON compatibility
gdf_bbox = gdf.to_crs(epsg=32760)

# 4. Export the bounding box as a GeoJSON file
gdf_bbox.to_file("2-dem-bbox.geojson", driver="GeoJSON")

print("Bounding box successfully exported as dem_bounding_box.geojson!")

Bounding box successfully exported as dem_bounding_box.geojson!


In [7]:
import rasterio

with rasterio.open("1-dawasamu-dem.tif") as src:
    print(f"CRS: {src.crs}")
    print(f"Is Projected: {src.crs.is_projected}")
    print(f"Linear Units: {src.crs.linear_units if src.crs else 'None'}")
    print(f"Pixel Resolution (X, Y): {src.res}")
    print(f"NoData Value: {src.nodatavals}")
    print(f"Data Type: {src.dtypes[0]}")

CRS: EPSG:32760
Is Projected: True
Linear Units: metre
Pixel Resolution (X, Y): (0.4725758554216868, 0.5418054419889503)
NoData Value: (0.0,)
Data Type: float64


In [11]:
import numpy as np
import rasterio

with rasterio.open("1-dawasamu-dem.tif") as src:
    dem = src.read(1)
    nodata = src.nodata

# 1. Calculate 2D gradient across the intact raster array
dy, dx = np.gradient(dem)

# 2. Identify flat pixels (where both horizontal and vertical gradient are 0)
flat_mask = (dx == 0) & (dy == 0)

# 3. Exclude nodata pixels from the flat pixel count
if nodata is not None:
    valid_mask = dem != nodata
    flat_pixels = np.sum(flat_mask & valid_mask)
    total_pixels = np.sum(valid_mask)
else:
    flat_pixels = np.sum(flat_mask)
    total_pixels = dem.size

print(f"Flat pixels percentage: {(flat_pixels / total_pixels) * 100:.2f}%")

Flat pixels percentage: 0.08%


In [14]:
import rasterio
from rasterio.enums import Resampling

input_path = "1-dawasamu-dem.tif"
output_path = "2-dem-resampled.tif"

target_resolution = 0.5  # 0.5 meter square pixels

with rasterio.open(input_path) as src:
    scale_x = src.res[0] / target_resolution
    scale_y = src.res[1] / target_resolution
    
    new_width = int(src.width * scale_x)
    new_height = int(src.height * scale_y)

    new_transform = rasterio.Affine(
        target_resolution, 0.0, src.transform.c,
        0.0, -target_resolution, src.transform.f
    )

    meta = src.meta.copy()
    meta.update({
        'driver': 'GTiff',
        'height': new_height,
        'width': new_width,
        'transform': new_transform,
        'dtype': 'float32'
    })

    data = src.read(
        1,
        out_shape=(new_height, new_width),
        resampling=Resampling.bilinear
    )

    with rasterio.open(output_path, 'w', **meta) as dst:
        dst.write(data, 1)

print("Resampled DEM saved to:", output_path)

Resampled DEM saved to: 2-dem-resampled.tif


#### 2. Land Use / Land Cover (LULC) MapFile Name in Script: dawasamu_lulc.tifFormat: Single-band GeoTIFF (.tif) containing integer codes.

What it represents: Spatial classification of the landscape (e.g., dense primary forest, shrubland, agriculture).Critical Requirement: Every integer code (pixel value) present in this map must have a corresponding row in your biophysical_table.csv.🌧


#### 5. Rainfall Erosivity (R Factor) File Name in Script: dawasamu_erosivity_r.tifFormat: Single-band GeoTIFF (.tif)  
What it represents: The climatic erosive power of rainfall.Units: MJ ⋅ mm ⋅ ha⁻¹ ⋅ h⁻¹ ⋅ yr⁻¹.Note: If spatial grid data is unavailable for Dawasamu, you can create a proxy raster where every pixel contains a uniform value derived from Fiji's standard annual rainfall equations

#### 6. Soil Erodibility (K Factor) File Name in Script: dawasamu_erodibility_k.tifFormat: 

Single-band GeoTIFF (.tif)What it represents: The susceptibility of different soil types to erosion.Units: t ⋅ ha ⋅ h ⋅ ha⁻¹ ⋅ MJ⁻¹ ⋅ mm⁻¹.

#### 7. Watershed / Catchment Boundary VectorFile Name in Script: dawasamu_catchment_boundary.shp  
Format: ESRI Shapefile components (.shp, .shx, .dbf, and .prj).What it represents: Polygon boundaries defining your area of interest. The model uses this layer to aggregate the pixel losses into a clean summary table of total sediment export tons per sub-catchment.

#### Quick Verification Checklist    
Before running your notebook script, ensure that: All rasters share matching pixel resolutions (e.g., 30m x 30m pixels). All files are named exactly as shown in your execution block parameters dictionary. Your separate biophysical_table.csv is saved in the parent /dawasamu/ folder, not hidden inside the data/ folder. Are your spatial layers currently available as GIS files, or would you like a script snippet to verify the projection (CRS) of your files automatically inside the notebook before launching the model?